# Delta Lake Incremental Processing & SCD Assignment

**Objective:** Perform incremental data processing using Delta Lake — load a customer master dataset into a Delta table, clean it, simulate an incremental batch of updates/inserts, and apply MERGE operations to update existing records and insert new ones, demonstrating both **SCD Type 1** (overwrite) and **SCD Type 2** (history-preserving) patterns.

**Engine:** [`deltalake`](https://delta-io.github.io/delta-rs/) (delta-rs Python bindings) — a native Delta Lake implementation that doesn't require Spark/JVM, used here for the same `MERGE` semantics as Databricks Delta Lake.

**Datasets:**
- `../data/customer_master.csv` — 151 customer records derived from the Superstore dataset (with a few nulls and one duplicate row deliberately included to exercise the cleaning step)
- `../data/customer_incremental.csv` — 15 records: 10 existing customers with changed `segment`/`city`, and 5 brand-new customers

## 1. Load dataset into a Delta table

In [1]:
import os
import pandas as pd
from deltalake import DeltaTable, write_deltalake

DATA_DIR = "../data"
DELTA_DIR = "../delta"
os.makedirs(DELTA_DIR, exist_ok=True)

master_df = pd.read_csv(f"{DATA_DIR}/customer_master.csv")
print("Raw master rows:", master_df.shape)
master_df.head()

Raw master rows: (151, 9)


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,2024-01-01
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,2024-01-01
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,2024-01-01
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,2024-01-01
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,2024-01-01


In [2]:
write_deltalake(f"{DELTA_DIR}/customer_table", master_df, mode="overwrite")
dt = DeltaTable(f"{DELTA_DIR}/customer_table")
print("Loaded into Delta table at", f"{DELTA_DIR}/customer_table")
print("Delta table version:", dt.version())
dt.to_pandas().head()

Loaded into Delta table at ../delta/customer_table
Delta table version: 0


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,2024-01-01
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,2024-01-01
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,2024-01-01
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,2024-01-01
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,2024-01-01


## 2. Basic cleaning: handle nulls, remove duplicates

In [3]:
df = dt.to_pandas()
print("Null counts per column:")
print(df.isnull().sum())
print("\nDuplicate rows (full-row duplicates):", df.duplicated().sum())
print("Duplicate customer_id rows:", df["customer_id"].duplicated().sum())

Null counts per column:
customer_id      0
customer_name    0
segment          1
country          0
city             1
state            0
postal_code      0
region           0
last_updated     0
dtype: int64

Duplicate rows (full-row duplicates): 1
Duplicate customer_id rows: 1


In [4]:
df_clean = df.copy()
df_clean["segment"] = df_clean["segment"].fillna("Unknown")
df_clean["city"] = df_clean["city"].fillna("Unknown")
df_clean = df_clean.drop_duplicates(subset=["customer_id"]).reset_index(drop=True)

print("Shape before cleaning:", df.shape)
print("Shape after cleaning:", df_clean.shape)
print("Remaining nulls:", df_clean.isnull().sum().sum())
print("Remaining duplicate customer_ids:", df_clean["customer_id"].duplicated().sum())

Shape before cleaning: (151, 9)
Shape after cleaning: (150, 9)
Remaining nulls: 0
Remaining duplicate customer_ids: 0


In [5]:
write_deltalake(f"{DELTA_DIR}/customer_table", df_clean, mode="overwrite")
dt = DeltaTable(f"{DELTA_DIR}/customer_table")
print("Cleaned data written back to the Delta table. New version:", dt.version())

Cleaned data written back to the Delta table. New version: 1


## 3. Create a second dataset simulating incremental data

In [6]:
incremental_df = pd.read_csv(f"{DATA_DIR}/customer_incremental.csv")
print("Incremental batch rows:", incremental_df.shape)
incremental_df

Incremental batch rows: (15, 9)


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,NP-18670,Nora Paige,Corporate,United States,Edmond (Updated),Oklahoma,73034,Central,2024-06-01
1,LC-16930,Linda Cazamias,Home Office,United States,Naperville (Updated),Illinois,60540,Central,2024-06-01
2,CB-12535,Claudia Bergmann,Home Office,United States,Chapel Hill (Updated),North Carolina,27514,South,2024-06-01
3,BM-11140,Becky Martin,Corporate,United States,San Antonio (Updated),Texas,78207,Central,2024-06-01
4,RD-19900,Ruben Dartt,Corporate,United States,Carlsbad (Updated),New Mexico,88220,West,2024-06-01
5,HM-14980,Henry MacAllister,Corporate,United States,New York City (Updated),New York,10009,East,2024-06-01
6,SH-19975,Sally Hughsby,Home Office,United States,San Francisco (Updated),California,94122,West,2024-06-01
7,AC-10420,Alyssa Crouse,Home Office,United States,San Francisco (Updated),California,94122,West,2024-06-01
8,JE-16165,Justin Ellison,Home Office,United States,Franklin (Updated),Wisconsin,53132,Central,2024-06-01
9,JK-15640,Jim Kriz,Consumer,United States,New York City (Updated),New York,10009,East,2024-06-01


## 4. SCD Type 1 MERGE — overwrite changed attributes, insert new customers

SCD1 keeps no history: matched rows are simply overwritten with the incoming values, and unmatched (new) rows are inserted.

In [7]:
dt_scd1 = DeltaTable(f"{DELTA_DIR}/customer_table")

all_cols = ["customer_id", "customer_name", "segment", "country", "city", "state", "postal_code", "region", "last_updated"]
update_map = {c: f"s.{c}" for c in all_cols}

(dt_scd1.merge(
    source=incremental_df,
    predicate="s.customer_id = t.customer_id",
    source_alias="s",
    target_alias="t")
 .when_matched_update(updates=update_map)
 .when_not_matched_insert(updates=update_map)
 .execute())

dt_scd1 = DeltaTable(f"{DELTA_DIR}/customer_table")
print("SCD1 merge complete. Table version:", dt_scd1.version())
dt_scd1.to_pandas().sort_values("customer_id").head(20)

SCD1 merge complete. Table version: 2


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
19,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,2024-01-01
149,AB-10060,Adam Bellavance,Home Office,United States,New York City,New York,10009,East,2024-01-01
12,AC-10420,Alyssa Crouse,Home Office,United States,San Francisco (Updated),California,94122,West,2024-06-01
70,AD-10180,Alan Dominguez,Home Office,United States,Houston,Texas,77041,Central,2024-01-01
23,AG-10270,Alejandro Grove,Consumer,United States,West Jordan,Utah,84084,West,2024-01-01
128,AG-10495,Andrew Gjertsen,Corporate,United States,Philadelphia,Pennsylvania,19140,East,2024-01-01
138,AG-10525,Andy Gerbode,Corporate,United States,Saint Petersburg,Florida,33710,South,2024-01-01
1,AG-10900,Arthur Gainer,Consumer,United States,Tucson,Arizona,85705,West,2024-06-01
130,AH-10195,Alan Haines,Corporate,United States,Tamarac,Florida,33319,South,2024-01-01
124,AH-10210,Alan Hwang,Consumer,United States,Brentwood,California,94513,West,2024-01-01


## 5. SCD Type 2 MERGE — preserve history of changes

SCD2 never overwrites a changed row in place. Instead it:
1. Marks the existing matching row as expired (`is_current = 0`, sets `effective_end_date`).
2. Inserts a brand-new row holding the updated values, with `is_current = 1` and a fresh `effective_start_date`.

(`is_current` is stored as an int flag rather than a boolean — delta-rs has a known bug computing MERGE statistics on boolean columns.)

This is built on a separate Delta table (`customer_table_scd2`) seeded from the cleaned master data, since it carries extra history-tracking columns.

In [8]:
import pyarrow as pa

scd2_base = df_clean.copy()
scd2_base["effective_start_date"] = "2024-01-01"
scd2_base["effective_end_date"] = pd.array([None] * len(scd2_base), dtype=object)
scd2_base["is_current"] = 1  # 1 = current, 0 = expired (int flag avoids a delta-rs boolean-stats bug in MERGE)

# Cast effective_end_date to a proper Arrow string type (all-null object columns
# would otherwise infer as an unsupported Arrow "null" type for Delta Lake)
scd2_table = pa.Table.from_pandas(scd2_base, preserve_index=False)
end_idx = scd2_table.schema.get_field_index("effective_end_date")
scd2_table = scd2_table.set_column(end_idx, "effective_end_date", pa.array([None] * len(scd2_base), type=pa.string()))

write_deltalake(f"{DELTA_DIR}/customer_table_scd2", scd2_table, mode="overwrite")
dt_scd2 = DeltaTable(f"{DELTA_DIR}/customer_table_scd2")
print("SCD2 table seeded. Version:", dt_scd2.version())

SCD2 table seeded. Version: 0


In [9]:
# Work out which incremental records actually changed an attribute we track for history
current_active = dt_scd2.to_pandas().query("is_current == 1")

compare = incremental_df.merge(current_active, on="customer_id", suffixes=("_new", "_old"))
changed_mask = (compare["segment_new"] != compare["segment_old"]) | (compare["city_new"] != compare["city_old"])
changed_ids = compare.loc[changed_mask, "customer_id"].tolist()
new_ids = incremental_df[~incremental_df["customer_id"].isin(current_active["customer_id"])]["customer_id"].tolist()

print("Existing customers with changed attributes:", changed_ids)
print("Brand-new customers to insert:", new_ids)

Existing customers with changed attributes: ['NP-18670', 'LC-16930', 'CB-12535', 'BM-11140', 'RD-19900', 'HM-14980', 'SH-19975', 'AC-10420', 'JE-16165', 'JK-15640']
Brand-new customers to insert: ['GT-14755', 'AG-10900', 'MM-18280', 'AR-10405', 'RA-19915']


In [10]:
# Step A: expire the current row for every customer whose attributes changed
expire_df = pd.DataFrame({"customer_id": changed_ids})

(dt_scd2.merge(
    source=expire_df,
    predicate="s.customer_id = t.customer_id and t.is_current = 1",
    source_alias="s",
    target_alias="t")
 .when_matched_update(updates={
     "is_current": "0",
     "effective_end_date": "'2024-06-01'",
 })
 .execute())

print("Expired", len(changed_ids), "outdated rows.")

Expired 10 outdated rows.


In [11]:
# Step B: insert fresh current-version rows for the changed customers + the brand-new ones
inserts = incremental_df[incremental_df["customer_id"].isin(changed_ids + new_ids)].copy()
inserts["effective_start_date"] = "2024-06-01"
inserts["effective_end_date"] = pd.array([None] * len(inserts), dtype=object)
inserts["is_current"] = 1

inserts_table = pa.Table.from_pandas(inserts, preserve_index=False)
end_idx = inserts_table.schema.get_field_index("effective_end_date")
inserts_table = inserts_table.set_column(end_idx, "effective_end_date", pa.array([None] * len(inserts), type=pa.string()))

write_deltalake(f"{DELTA_DIR}/customer_table_scd2", inserts_table, mode="append")

dt_scd2 = DeltaTable(f"{DELTA_DIR}/customer_table_scd2")
print("SCD2 merge complete. Table version:", dt_scd2.version())

history_view = dt_scd2.to_pandas()
history_view[history_view["customer_id"].isin(changed_ids)].sort_values(["customer_id", "effective_start_date"])

SCD2 merge complete. Table version: 2


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated,effective_start_date,effective_end_date,is_current
22,AC-10420,Alyssa Crouse,Corporate,United States,San Francisco,California,94122,West,2024-01-01,2024-01-01,2024-06-01,0
7,AC-10420,Alyssa Crouse,Home Office,United States,San Francisco (Updated),California,94122,West,2024-06-01,2024-06-01,None,1
18,BM-11140,Becky Martin,Consumer,United States,San Antonio,Texas,78207,Central,2024-01-01,2024-01-01,2024-06-01,0
3,BM-11140,Becky Martin,Corporate,United States,San Antonio (Updated),Texas,78207,Central,2024-06-01,2024-06-01,None,1
17,CB-12535,Claudia Bergmann,Corporate,United States,Chapel Hill,North Carolina,27514,South,2024-01-01,2024-01-01,2024-06-01,0
2,CB-12535,Claudia Bergmann,Home Office,United States,Chapel Hill (Updated),North Carolina,27514,South,2024-06-01,2024-06-01,None,1
20,HM-14980,Henry MacAllister,Consumer,United States,New York City,New York,10009,East,2024-01-01,2024-01-01,2024-06-01,0
5,HM-14980,Henry MacAllister,Corporate,United States,New York City (Updated),New York,10009,East,2024-06-01,2024-06-01,None,1
23,JE-16165,Justin Ellison,Corporate,United States,Franklin,Wisconsin,53132,Central,2024-01-01,2024-01-01,2024-06-01,0
8,JE-16165,Justin Ellison,Home Office,United States,Franklin (Updated),Wisconsin,53132,Central,2024-06-01,2024-06-01,None,1


## 6. Validate results (row counts, duplicates)

In [12]:
final_scd1 = DeltaTable(f"{DELTA_DIR}/customer_table").to_pandas()
final_scd2 = DeltaTable(f"{DELTA_DIR}/customer_table_scd2").to_pandas()

print("--- SCD1 table ---")
print("Row count:", len(final_scd1))
print("Expected: 150 original + 5 new = 155")
print("Duplicate customer_ids:", final_scd1["customer_id"].duplicated().sum())

print("\n--- SCD2 table ---")
print("Total row count (including history):", len(final_scd2))
print("Current rows (is_current=1):", int((final_scd2["is_current"] == 1).sum()))
print("Expected current rows: 150 original + 5 new = 155")
print("Duplicate customer_ids among CURRENT rows:",
      final_scd2[final_scd2["is_current"] == 1]["customer_id"].duplicated().sum())
print("History rows (is_current=0):", int((final_scd2["is_current"] == 0).sum()), "(expected: 10 expired versions)")

--- SCD1 table ---
Row count: 155
Expected: 150 original + 5 new = 155
Duplicate customer_ids: 0

--- SCD2 table ---
Total row count (including history): 165
Current rows (is_current=1): 155
Expected current rows: 150 original + 5 new = 155
Duplicate customer_ids among CURRENT rows: 0
History rows (is_current=0): 10 (expected: 10 expired versions)


## 7. Final dataset & summary

In [13]:
print("Final SCD1 (current-state) table:")
final_scd1.sort_values("customer_id").reset_index(drop=True)

Final SCD1 (current-state) table:


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated
0,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,2024-01-01
1,AB-10060,Adam Bellavance,Home Office,United States,New York City,New York,10009,East,2024-01-01
2,AC-10420,Alyssa Crouse,Home Office,United States,San Francisco (Updated),California,94122,West,2024-06-01
3,AD-10180,Alan Dominguez,Home Office,United States,Houston,Texas,77041,Central,2024-01-01
4,AG-10270,Alejandro Grove,Consumer,United States,West Jordan,Utah,84084,West,2024-01-01
...,...,...,...,...,...,...,...,...,...
150,VB-21745,Victoria Brennan,Corporate,United States,Columbus,Georgia,31907,South,2024-01-01
151,VD-21670,Valerie Dominguez,Consumer,United States,Columbia,Tennessee,38401,South,2024-01-01
152,VM-21685,Valerie Mitchum,Home Office,United States,Westfield,New Jersey,7090,East,2024-01-01
153,VW-21775,Victoria Wilson,Corporate,United States,Medina,Ohio,44256,East,2024-01-01


In [14]:
print("Final SCD2 table — full history for the customers that changed:")
final_scd2[final_scd2["customer_id"].isin(changed_ids)].sort_values(["customer_id", "effective_start_date"]).reset_index(drop=True)

Final SCD2 table — full history for the customers that changed:


,customer_id,customer_name,segment,country,city,state,postal_code,region,last_updated,effective_start_date,effective_end_date,is_current
0,AC-10420,Alyssa Crouse,Corporate,United States,San Francisco,California,94122,West,2024-01-01,2024-01-01,2024-06-01,0
1,AC-10420,Alyssa Crouse,Home Office,United States,San Francisco (Updated),California,94122,West,2024-06-01,2024-06-01,None,1
2,BM-11140,Becky Martin,Consumer,United States,San Antonio,Texas,78207,Central,2024-01-01,2024-01-01,2024-06-01,0
3,BM-11140,Becky Martin,Corporate,United States,San Antonio (Updated),Texas,78207,Central,2024-06-01,2024-06-01,None,1
4,CB-12535,Claudia Bergmann,Corporate,United States,Chapel Hill,North Carolina,27514,South,2024-01-01,2024-01-01,2024-06-01,0
5,CB-12535,Claudia Bergmann,Home Office,United States,Chapel Hill (Updated),North Carolina,27514,South,2024-06-01,2024-06-01,None,1
6,HM-14980,Henry MacAllister,Consumer,United States,New York City,New York,10009,East,2024-01-01,2024-01-01,2024-06-01,0
7,HM-14980,Henry MacAllister,Corporate,United States,New York City (Updated),New York,10009,East,2024-06-01,2024-06-01,None,1
8,JE-16165,Justin Ellison,Corporate,United States,Franklin,Wisconsin,53132,Central,2024-01-01,2024-01-01,2024-06-01,0
9,JE-16165,Justin Ellison,Home Office,United States,Franklin (Updated),Wisconsin,53132,Central,2024-06-01,2024-06-01,None,1


### Summary

This notebook walks through a fairly typical incremental-loading scenario for a customer master table, built on Delta Lake's `MERGE` operation rather than Spark SQL, using the `deltalake` (delta-rs) Python package.

The starting point was 151 customer rows pulled from the Superstore dataset, with a couple of missing `segment`/`city` values and one duplicate record baked in on purpose so the cleaning step had something real to do — nulls were filled with `"Unknown"` and the duplicate was dropped, leaving 150 clean rows that became the initial Delta table.

From there, a 15-row incremental batch was merged in: 10 existing customers had their segment or city changed, and 5 were entirely new. Two different merge strategies were applied to the same incremental batch to show the difference in practice. The **SCD1** version simply overwrote the 10 changed rows in place and appended the 5 new ones, landing at 155 rows with no trace of what the old values used to be — fine when you only ever care about the current state. The **SCD2** version handled the same batch very differently: instead of overwriting, it expired the 10 outdated rows (flipping `is_current` to `False` and stamping an `effective_end_date`) and inserted fresh rows carrying the new values, so the table now holds 165 rows total — 155 current plus 10 expired historical versions — and you can still see exactly what each changed customer's record looked like before and after.

Validation confirmed both tables behaved as expected: no duplicate `customer_id`s in either the SCD1 table or among SCD2's current rows, and the SCD2 history count matched the number of customers that actually changed. The practical takeaway: SCD1 is simpler and is the right call when history doesn't matter, while SCD2 costs more storage and merge complexity but is the only option when you need to answer "what did this look like before the update?" — which is exactly the kind of question Delta Lake's transaction log and versioned MERGE operations are built to support.